In [1]:
import os

from google import genai

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

C:\Users\Dell\AppData\Local\Temp\ipykernel_5164\4208007953.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


**Load PDFs**

In [2]:
folder = "all-pdfs"

documents = []

for file in os.listdir(folder):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(folder, file))
        documents.extend(loader.load())

print(len(documents))

121


**Split Documents**

In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = splitter.split_documents(documents)

print(len(chunks))

530


**Create Embeddings**

In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

**Create FAISS database**

In [5]:
vector_db = FAISS.from_documents(
    chunks,
    embeddings
)

**Cretae BM25**

In [6]:
bm25 = BM25Retriever.from_documents(chunks)

bm25.k = 4

**Create FAISS Retriever**

In [7]:
faiss = vector_db.as_retriever(
    search_kwargs={"k":4}
)

**Create Enssemble Retriever**

In [8]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25, faiss],
    weights=[0.5,0.5]
)

**Gemini client**

In [9]:
client = genai.Client(
    api_key="your_api_key"
)

**Create conversational memory**

In [10]:
chat_history = []

**Build chatBot function**

In [11]:
def ask_question(question):

    docs = ensemble_retriever.invoke(question)

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    history = "\n".join(chat_history)

    prompt = f"""
You are an AI assistant.

Previous Conversation:
{history}

Answer ONLY using the context below.

If the answer is not available, reply:
"I don't know based on the provided context."

If the user asks a follow-up question, use the previous conversation when necessary.

Keep your answer concise (2–4 sentences).

Context:
{context}

Question:
{question}

Answer:
"""

    try:

        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )

        answer = response.text

    except Exception as e:

        print("Error:", e)

        answer = "Gemini server is currently busy. Please try again later."

    chat_history.append(f"User: {question}")
    chat_history.append(f"Assistant: {answer}")

    return answer

**test ChatBot**

In [12]:
print(ask_question("What is BERT?"))

BERT is a framework that involves two steps: pre-training on unlabeled data and fine-tuning with labeled data for downstream tasks. The model is initialized with pre-trained parameters and then fine-tuned. BERT uses a bidirectional Transformer architecture, meaning its representations are jointly conditioned on both left and right context in all layers.


In [13]:
print(ask_question("Who developed it?"))

Error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 55.771355077s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'}, 'quot

**Display memory**

In [14]:
for msg in chat_history:
    print(msg)

User: What is BERT?
Assistant: BERT is a framework that involves two steps: pre-training on unlabeled data and fine-tuning with labeled data for downstream tasks. The model is initialized with pre-trained parameters and then fine-tuned. BERT uses a bidirectional Transformer architecture, meaning its representations are jointly conditioned on both left and right context in all layers.
User: Who developed it?
Assistant: Gemini server is currently busy. Please try again later.


**clear memory**

In [15]:
chat_history.clear()

print(chat_history)

[]


In [16]:
for msg in chat_history:
    print(msg)